# METRICAS #

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuración visual para los gráficos
plt.style.use('ggplot')

# Celda 2: Definición de las Funciones de Métricas #

In [ ]:
def hit_rate_at_k(recomendaciones, items_relevantes, k):
    """Calcula el HR@k."""
    aciertos = 0
    total_consultas = len(recomendaciones)
    for i in range(total_consultas):
        rec_k = recomendaciones[i][:k]
        if any(item in rec_k for item in items_relevantes[i]):
            aciertos += 1
    return aciertos / total_consultas if total_consultas > 0 else 0.0

def precision_at_k(recomendaciones, items_relevantes, k):
    """Calcula Precision@K (Relevantes en Top-K / K)."""
    precisiones = []
    for recs, rels in zip(recomendaciones, items_relevantes):
        recs_k = recs[:k]
        relevantes_en_k = len(set(recs_k).intersection(set(rels)))
        precisiones.append(relevantes_en_k / k)
    return np.mean(precisiones)

def recall_at_k(recomendaciones, items_relevantes, k):
    """Calcula Recall@K (Relevantes en Top-K / Total Relevantes)."""
    recalls = []
    for recs, rels in zip(recomendaciones, items_relevantes):
        recs_k = recs[:k]
        relevantes_en_k = len(set(recs_k).intersection(set(rels)))
        total_relevantes = len(rels)
        if total_relevantes > 0:
            recalls.append(relevantes_en_k / total_relevantes)
        else:
            recalls.append(0.0)
    return np.mean(recalls)

def calcular_dcg_at_k(recs_k, dict_relevancia):
    """Función auxiliar para calcular DCG."""
    dcg = 0.0
    for i, item in enumerate(recs_k):
        rel_i = dict_relevancia.get(item, 0)
        dcg += rel_i / np.log2(i + 2)
    return dcg

def ndcg_at_k(recomendaciones, diccionarios_relevancia, k):
    """Calcula NDCG@K."""
    ndcgs = []
    for recs, dict_rel in zip(recomendaciones, diccionarios_relevancia):
        recs_k = recs[:k]
        dcg = calcular_dcg_at_k(recs_k, dict_rel)
        items_ideales = sorted(dict_rel.keys(), key=lambda x: dict_rel[x], reverse=True)[:k]
        idcg = calcular_dcg_at_k(items_ideales, dict_rel)
        if idcg > 0:
            ndcgs.append(dcg / idcg)
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)

# Celda 3: Carga de Datos Limpios y Predicciones de los Modelos #

In [ ]:
# 1. Cargar el dataset de testing limpio
df_test = pd.read_csv("../Data/2020-Apr-L.csv")

# ==============================================================================
# IMPORTANTE: Aquí debes cargar los resultados exportados por ml.ipynb y dl.ipynb
# Por ejemplo, podrías haber guardado las recomendaciones en archivos CSV o JSON:
# preds_ml = pd.read_csv("predicciones_ml.csv")
# preds_dl = pd.read_csv("predicciones_dl.csv")
# preds_hibrido = pd.read_csv("predicciones_hibrido.csv")
# ==============================================================================

# Variables de simulación para el pipeline de evaluación (Reemplazar con datos reales)
K = 5